# SupplyPrescript
## Notebook 8: Closed-Loop Analytics & Decision Logging

### Objective
Notebook 8 introduces the closed-loop analytics component of SupplyPrescript. It records predictions, recommendations, manager decisions, actual outcomes, and feedback for future improvement.

### Closed-Loop Flow
Prediction → Recommendation → Manager Decision → Actual Outcome → Evaluation → Feedback Dataset → Future Improvement

### Important Note
The current project does not contain real historical manager decisions or operational outcomes. Therefore, demonstration values are generated for testing and are clearly labelled as prototype data.


## 8.1 Import Required Libraries


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")


## 8.2 Define Project Paths


In [ ]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

print("Project directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)


## 8.3 Load Notebook 7 Decision Results


In [ ]:
INPUT_PATH = os.path.join(PROCESSED_DIR, "SupplyPrescript_decision_results.csv")

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Decision results not found at: {INPUT_PATH}. Run Notebook 7 first."
    )

decision_results = pd.read_csv(INPUT_PATH)
print("Decision results shape:", decision_results.shape)
print(decision_results.columns.tolist())
decision_results.head()


## 8.4 Inspect the Existing Decision Data


In [ ]:
print(decision_results.info())
print("\nMissing values:")
print(decision_results.isnull().sum())


## 8.5 Create the Closed-Loop Decision Log


In [ ]:
decision_log = decision_results.copy()

decision_log["manager_decision"] = decision_log["recommended_action"]
decision_log["actual_cost"] = decision_log["additional_cost"]
decision_log["actual_delay_days"] = np.nan
decision_log["outcome_status"] = "Pending"
decision_log["recommendation_followed"] = True
decision_log["cost_difference"] = (
    decision_log["actual_cost"] - decision_log["additional_cost"]
)

decision_log.head()


## 8.6 Generate Demonstration Outcome Data


In [ ]:
rng = np.random.default_rng(42)

decision_log["actual_delay_days"] = rng.integers(0, 6, size=len(decision_log))
decision_log["outcome_status"] = np.where(
    decision_log["actual_delay_days"] <= 1, "Successful", "Delayed"
)

cost_variation = rng.integers(-1000, 3001, size=len(decision_log))
decision_log["actual_cost"] = (
    decision_log["additional_cost"] + cost_variation
).clip(lower=0)

decision_log["cost_difference"] = (
    decision_log["actual_cost"] - decision_log["additional_cost"]
)

decision_log.head()


## 8.7 Simulate Manager Decisions for Testing


In [ ]:
decision_log["manager_decision"] = decision_log["recommended_action"]

alternative_actions = [
    "standard_shipping",
    "air_freight",
    "alternative_supplier",
    "delay_launch"
]

for index in range(0, len(decision_log), 5):
    recommended = decision_log.loc[index, "recommended_action"]
    alternatives = [a for a in alternative_actions if a != recommended]
    decision_log.loc[index, "manager_decision"] = alternatives[0]

decision_log["recommendation_followed"] = (
    decision_log["manager_decision"] == decision_log["recommended_action"]
)

decision_log[[
    "shipment_index", "recommended_action", "manager_decision",
    "recommendation_followed"
]].head(10)


## 8.8 Evaluate Whether Recommendations Were Followed


In [ ]:
followed_count = int(decision_log["recommendation_followed"].sum())
not_followed_count = len(decision_log) - followed_count
followed_percentage = followed_count / len(decision_log) * 100

print("Recommendations followed    :", followed_count)
print("Recommendations not followed:", not_followed_count)
print(f"Followed percentage         : {followed_percentage:.2f}%")


## 8.9 Evaluate Operational Outcomes


In [ ]:
successful_count = int((decision_log["outcome_status"] == "Successful").sum())
delayed_count = int((decision_log["outcome_status"] == "Delayed").sum())
success_rate = successful_count / len(decision_log) * 100

print("Successful outcomes:", successful_count)
print("Delayed outcomes   :", delayed_count)
print(f"Success rate       : {success_rate:.2f}%")


## 8.10 Analyze Cost Difference


In [ ]:
average_estimated_cost = decision_log["additional_cost"].mean()
average_actual_cost = decision_log["actual_cost"].mean()
average_cost_difference = decision_log["cost_difference"].mean()

print(f"Average estimated cost : ${average_estimated_cost:,.2f}")
print(f"Average actual cost    : ${average_actual_cost:,.2f}")
print(f"Average cost difference: ${average_cost_difference:,.2f}")


## 8.11 Identify Successful Recommendations


In [ ]:
decision_log["recommendation_success"] = (
    decision_log["recommendation_followed"]
    & (decision_log["actual_delay_days"] <= 1)
)

print("Successful recommendations:", int(decision_log["recommendation_success"].sum()))

decision_log[[
    "recommended_action", "manager_decision",
    "actual_delay_days", "recommendation_success"
]].head(10)


## 8.12 Analyze Recommendations by Action


In [ ]:
action_summary = (
    decision_log.groupby("recommended_action")
    .agg(
        recommendations=("recommended_action", "count"),
        followed=("recommendation_followed", "sum"),
        successful=("recommendation_success", "sum"),
        average_actual_delay=("actual_delay_days", "mean"),
        average_actual_cost=("actual_cost", "mean"),
        average_cost_difference=("cost_difference", "mean")
    )
    .reset_index()
)

action_summary["follow_through_rate"] = (
    action_summary["followed"] / action_summary["recommendations"] * 100
)
action_summary["success_rate"] = (
    action_summary["successful"] / action_summary["recommendations"] * 100
)

action_summary


## 8.13 Visualize Recommendation Success


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=action_summary, x="recommended_action", y="success_rate")
plt.xlabel("Recommended Action")
plt.ylabel("Success Rate (%)")
plt.title("Recommendation Success Rate by Action")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 8.14 Visualize Actual Delay by Recommended Action


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=action_summary, x="recommended_action", y="average_actual_delay")
plt.xlabel("Recommended Action")
plt.ylabel("Average Actual Delay (Days)")
plt.title("Average Actual Delay by Recommended Action")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 8.15 Generate Feedback Records


In [ ]:
feedback_columns = [
    "shipment_index", "delay_probability", "risk_level",
    "recommended_action", "manager_decision", "additional_cost",
    "actual_cost", "cost_difference", "expected_delay_risk",
    "actual_delay_days", "outcome_status",
    "recommendation_followed", "recommendation_success"
]

feedback_data = decision_log[feedback_columns].copy()
feedback_data.head(10)


## 8.16 Identify Cases for Future Review


In [ ]:
review_cases = feedback_data[
    (~feedback_data["recommendation_followed"])
    | (~feedback_data["recommendation_success"])
]

print("Cases requiring review:", len(review_cases))
review_cases.head(10)


## 8.17 Create a Closed-Loop Performance Summary


In [ ]:
closed_loop_summary = pd.DataFrame({
    "Metric": [
        "Total Decisions", "Recommendations Followed",
        "Recommendations Not Followed", "Successful Recommendations",
        "Unsuccessful Recommendations", "Recommendation Follow-Through Rate (%)",
        "Overall Success Rate (%)", "Average Estimated Cost",
        "Average Actual Cost", "Average Cost Difference",
        "Average Actual Delay (Days)"
    ],
    "Value": [
        len(decision_log), followed_count, not_followed_count,
        int(decision_log["recommendation_success"].sum()),
        int((~decision_log["recommendation_success"]).sum()),
        followed_percentage, success_rate, average_estimated_cost,
        average_actual_cost, average_cost_difference,
        decision_log["actual_delay_days"].mean()
    ]
})

closed_loop_summary


## 8.18 Save the Closed-Loop Decision Log


In [ ]:
DECISION_LOG_PATH = os.path.join(PROCESSED_DIR, "closed_loop_decision_log.csv")
os.makedirs(PROCESSED_DIR, exist_ok=True)
decision_log.to_csv(DECISION_LOG_PATH, index=False)
print("Decision log saved to:")
print(DECISION_LOG_PATH)


## 8.19 Save the Feedback Dataset


In [ ]:
FEEDBACK_PATH = os.path.join(PROCESSED_DIR, "closed_loop_feedback.csv")
feedback_data.to_csv(FEEDBACK_PATH, index=False)
print("Feedback dataset saved to:")
print(FEEDBACK_PATH)


## 8.20 Save the Action Summary


In [ ]:
ACTION_SUMMARY_PATH = os.path.join(PROCESSED_DIR, "action_performance_summary.csv")
action_summary.to_csv(ACTION_SUMMARY_PATH, index=False)
print("Action summary saved to:")
print(ACTION_SUMMARY_PATH)


## 8.21 Validate Saved Files


In [ ]:
for path in [DECISION_LOG_PATH, FEEDBACK_PATH, ACTION_SUMMARY_PATH]:
    print(path, "->", os.path.exists(path))


# 8.22 Closed-Loop Analytics Interpretation

The prototype stores the full decision lifecycle:

1. **Prediction** — XGBoost estimates delay probability.
2. **Recommendation** — PuLP selects an operational action.
3. **Human Decision** — the manager decision is recorded.
4. **Actual Outcome** — actual delay and cost are recorded.
5. **Evaluation** — the recommendation is compared with the outcome.
6. **Feedback** — the resulting record becomes feedback for future improvement.

This provides the foundation for continuous improvement without falsely claiming that automatic retraining is already implemented.


# Summary

Notebook 8 completes the prototype closed-loop analytics layer of SupplyPrescript.

### Outputs
- `closed_loop_decision_log.csv`
- `closed_loop_feedback.csv`
- `action_performance_summary.csv`

### Project Flow
**Predict → Prescribe → Decide → Observe → Evaluate → Learn**

### Limitation
Manager decisions, actual costs, and actual delay values in this notebook are demonstration data because the current project does not yet contain a real operational feedback database. In production, these fields should come from a backend database, ERP system, or user interface.

### Future Enhancement
After sufficient real-world feedback is collected, the system can periodically recalibrate optimization parameters and retrain the predictive model.


# End of Notebook 8
